In [ ]:
import statistics
import warnings
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import regex
from plotly.colors import sample_colorscale
from plotly.express import colors
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from datetime import datetime
from pathlib import Path

from src.utils.util import range_normalization, sortElements
from src.visualizacion.color_maps import sample_random_colors
import dash
from dash import dcc, html
from dash import dash_table
import re
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import os
import time


import plotly.io as pio

from src.api import getInfoEstacionesComerciales
from src.api.APIs import getInfoFiabilidadEstacion
from src.utils import formatTimedelta  # loadViasFromTopos,
from src.utils import (
    dateFromText,
    getFilesByDate,
    guardarExcel,
    isEmpty,
    rellenarId,
    sortStrNumbers,
)
from src.visualizacion.visualizaciones import (
    build_hierarchical_dataframe,
    mostrarConfusionSankey,
    mostrarConfusionTree,
    setHoverInfo,
    setLayout,
    # mostrarConfusionSankey1
)

color24 = colors.qualitative.Dark24
color12 = colors.qualitative.Set3
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from pathlib import Path
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.units import inch
from datetime import datetime
from reportlab.lib.colors import Color
from datetime import timedelta
from reportlab.platypus import PageBreak


In [ ]:
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import regex
from plotly.colors import sample_colorscale
from plotly.express import colors
from plotly.subplots import make_subplots
from tqdm.auto import tqdm

from src.api import getInfoEstacionesComerciales
from src.api.APIs import getInfoFiabilidadEstacion
from src.utils import formatTimedelta  # loadViasFromTopos,
from src.utils import (
    dateFromText,
    getFilesByDate,
    guardarExcel,
    isEmpty,
    rellenarId,
    sortStrNumbers,
)
from src.visualizacion.visualizaciones import (
    build_hierarchical_dataframe,
    mostrarConfusionSankey,
    mostrarConfusionTree,
    setHoverInfo,
    setLayout,
    # mostrarConfusionSankey1
)

color24 = colors.qualitative.Dark24
color12 = colors.qualitative.Set3
from reportlab.lib.pagesizes import letter, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.utils import ImageReader
from pathlib import Path
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.units import inch
from datetime import datetime
from reportlab.lib.colors import Color
from datetime import timedelta
from reportlab.platypus import PageBreak


### Carga datos

In [ ]:
fecha_ini =  datetime.now().strftime("%Y-%m-%d")
# fecha_ini = "2026-03-11"
fecha_fin = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")
# fecha_fin = "2026-03-12"
fuentes = getFilesByDate(Path("data/FuentesVías"), fecha_ini, fecha_fin, ftype="xlsx")
resumen = []
computo_estacion = []

fuentes_vias = []
detalle_tren = []


with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with tqdm(total=len(fuentes)) as pbar:
        for f, _ in fuentes:
            fecha = dateFromText(f.stem)
            pbar.set_description(f"{fecha}")

            excel_info = pd.read_excel(f, sheet_name=None, engine="openpyxl")

            r = excel_info["Resumen"].copy()
            r.columns = r.iloc[0].tolist()

            r = r.loc[1:, r.columns[1:]]
            r["Fecha"] = fecha
            resumen.append(r)

            ce = excel_info["CómputoEstación"].copy()
            ce.columns = ce.iloc[2].tolist()

            ce = ce.loc[3:, ce.columns[1:]]
            ce["Fecha"] = fecha
            computo_estacion.append(ce)

            ev = excel_info["CómputoEstaciónVía"].copy()
            ev.columns = ev.iloc[2].tolist()

            ev = ev.loc[3:, ev.columns[1:]]
            ev["Fecha"] = fecha
            fuentes_vias.append(ev)

            dt = excel_info["DetalleTren"].copy()
            dt.columns = dt.iloc[0].tolist()

            dt = dt.loc[1:, dt.columns[1:]]
            detalle_tren.append(dt)

            pbar.update()


resumen = pd.concat(resumen).reset_index(drop=True)
computo_estacion = pd.concat(computo_estacion).reset_index(drop=True)
fuentes_vias = pd.concat(fuentes_vias).reset_index(drop=True)
detalle_tren = pd.concat(detalle_tren).reset_index(drop=True)
detalle_tren[["Cod. Est", "Tren"]] = detalle_tren[["Cod. Est", "Tren"]].map(rellenarId)

In [ ]:
fecha_ini =  (datetime.now() - timedelta(days=7)).strftime("%Y-%m-%d")
fecha_fin = datetime.now().strftime("%Y-%m-%d")
# fecha_base = datetime.strptime("2026-04-11", "%Y-%m-%d")
# fecha_ini =  (fecha_base - timedelta(days=7)).strftime("%Y-%m-%d")
# fecha_fin = "2026-04-11"
fuentes = getFilesByDate(Path("data/FuentesVías"), fecha_ini, fecha_fin, ftype="xlsx")
resumen_semanal = []
computo_estacion_semanal = []

fuentes_vias_semanal = []
detalle_tren_semanal = []


with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with tqdm(total=len(fuentes)) as pbar:
        for f, _ in fuentes:
            fecha = dateFromText(f.stem)
            pbar.set_description(f"{fecha}")

            excel_info = pd.read_excel(f, sheet_name=None, engine="openpyxl")

            r = excel_info["Resumen"].copy()
            r.columns = r.iloc[0].tolist()

            r = r.loc[1:, r.columns[1:]]
            r["Fecha"] = fecha
            resumen_semanal.append(r)

            ce = excel_info["CómputoEstación"].copy()
            ce.columns = ce.iloc[2].tolist()

            ce = ce.loc[3:, ce.columns[1:]]
            ce["Fecha"] = fecha
            computo_estacion_semanal.append(ce)

            ev = excel_info["CómputoEstaciónVía"].copy()
            ev.columns = ev.iloc[2].tolist()

            ev = ev.loc[3:, ev.columns[1:]]
            ev["Fecha"] = fecha
            fuentes_vias_semanal.append(ev)

            dt = excel_info["DetalleTren"].copy()
            dt.columns = dt.iloc[0].tolist()

            dt = dt.loc[1:, dt.columns[1:]]
            detalle_tren_semanal.append(dt)

            pbar.update()


resumen_semanal = pd.concat(resumen_semanal).reset_index(drop=True)
computo_estacion_semanal = pd.concat(computo_estacion_semanal).reset_index(drop=True)
fuentes_vias_semanal = pd.concat(fuentes_vias_semanal).reset_index(drop=True)
detalle_tren_semanal = pd.concat(detalle_tren_semanal).reset_index(drop=True)
detalle_tren_semanal[["Cod. Est", "Tren"]] = detalle_tren_semanal[["Cod. Est", "Tren"]].map(rellenarId)

#### Detalle por decil

In [ ]:
resumen_detallado = computo_estacion[["Fecha", "% Registro Vía CTC"]].copy()
vals = np.ceil((resumen_detallado["% Registro Vía CTC"] * 100).astype(int).values)

In [ ]:
resumen_detallado_semanal = computo_estacion_semanal[["Fecha", "% Registro Vía CTC"]].copy()
vals_semanal = np.ceil((resumen_detallado_semanal["% Registro Vía CTC"] * 100).astype(int).values)

In [ ]:
# Asignar decil
resumen_detallado = computo_estacion[["Fecha", "% Registro Vía CTC"]].copy()
vals = np.ceil((resumen_detallado["% Registro Vía CTC"] * 100).astype(int).values)
group = 20
r = np.round(vals, 2)
rest = r % group
result = np.where(vals == 100, r + group, r)
result = np.where(np.invert(result == 0), r + group - rest, r)
resumen_detallado["grupo"] = result

# Agrupar por decil
resumen_detallado = (
    resumen_detallado.groupby("Fecha")
    .agg({"grupo": lambda x: Counter(x)})["grupo"]
    .apply(pd.Series)
    .fillna(0)
    .astype(int)
)


cols = sorted(resumen_detallado.columns.astype(int))
resumen_detallado = resumen_detallado[cols]
new_cols = []
new_cols.append(cols[0])
for c in range(len(cols[1:-1])):
    new_cols.append(f"{cols[c]}-{cols[c+1]}")
new_cols.append(cols[-1] - group)
resumen_detallado.columns = [f"{c}%" for c in new_cols]

In [ ]:
# Asignar decil semanal
resumen_detallado_semanal = computo_estacion_semanal[["Fecha", "% Registro Vía CTC"]].copy()
vals_semanal= np.ceil((resumen_detallado_semanal["% Registro Vía CTC"] * 100).astype(int).values)
group_semanal = 20
r_semanal = np.round(vals_semanal, 2)
rest_semanal = r_semanal % group_semanal
result_semanal = np.where(vals_semanal == 100, r_semanal + group_semanal, r_semanal)
result_semanal = np.where(np.invert(result_semanal == 0), r_semanal + group_semanal - rest_semanal, r_semanal)
resumen_detallado_semanal["grupo"] = result_semanal

# Agrupar por decil
resumen_detallado_semanal = (
    resumen_detallado_semanal.groupby("Fecha")
    .agg({"grupo": lambda x: Counter(x)})["grupo"]
    .apply(pd.Series)
    .fillna(0)
    .astype(int)
)


cols_semanal = sorted(resumen_detallado_semanal.columns.astype(int))
resumen_detallado_semanal = resumen_detallado_semanal[cols_semanal]
new_cols_semanal = []
new_cols_semanal.append(cols_semanal[0])
for c in range(len(cols_semanal[1:-1])):
    new_cols_semanal.append(f"{cols_semanal[c]}-{cols_semanal[c+1]}")
new_cols_semanal.append(cols_semanal[-1] - group_semanal)
resumen_detallado_semanal.columns = [f"{c}%" for c in new_cols]

In [ ]:
traces = []
use_df = resumen_detallado.reset_index().copy()
# use_df["hover_info"] = setHoverInfo(use_df, use_df.columns[1:])
cmap_est = dict(zip(use_df.columns[1:], color24))
for c in use_df.columns[1:]:
    aux_df = use_df[["Fecha", c]]
    color = cmap_est.get(c)
    traces.extend(
        [
            go.Scatter(
                x=aux_df["Fecha"],
                y=aux_df[c],
                # hoverinfo="text",
                visible=True,
                showlegend=True,
                name=c,
                legendgroup=c,
                mode="lines+markers",
                marker_color=color,
                line_color=color,
                marker_opacity=0.25,
            )
        ]
    )

layout = setLayout("togglegroup", title="Evolución Registro Vía CTC")
layout.update(
    xaxis_title="Fecha",
    yaxis_title="Número estaciones",
    legend_title="% Registro Vía CTC",
    hovermode="x unified",
    height=600,
)

fig = go.Figure(traces, layout)
fig = fig.update_traces(
    # hovertext=setHoverInfo(use_df, use_df.columns[1:]).values,
    # hoverinfo="text",
    hoverinfo="name+y",
    # hovertemplate=f"%{{y}}"
)
# fig.show()

### Vías Fiables IHM

In [ ]:
actualizar_fiabilidad = False
if not Path("data/FiabilidadEstaciones.xlsx").exists() or actualizar_fiabilidad:
    fiabilidad = []
    estaciones = getInfoEstacionesComerciales()
    estaciones = pd.DataFrame(estaciones)
    for c in tqdm(estaciones[estaciones["commercial"]]["code"].unique()):
        fiab = getInfoFiabilidadEstacion(c)
        fiab["Código"] = c
        fiabilidad.append(fiab)
    fiabilidad = pd.concat(fiabilidad)
    guardarExcel(fiabilidad, "data/FiabilidadEstaciones.xlsx", append_sheet=False)

else:
    fiabilidad = pd.read_excel("data/FiabilidadEstaciones.xlsx")
    fiabilidad["Código"] = fiabilidad["Código"].astype(str).apply(rellenarId)

In [ ]:
actualizar_fiabilidad_semanal = False
if not Path("data/FiabilidadEstaciones.xlsx").exists() or actualizar_fiabilidad_semanal:
    print("1")
    fiabilidad_semanal = []
    estaciones = getInfoEstacionesComerciales()
    estaciones = pd.DataFrame(estaciones)
    for c in tqdm(estaciones[estaciones["commercial"]]["code"].unique()):
        fiab = getInfoFiabilidadEstacion(c)
        fiab["Código"] = c
        fiabilidad_semanal.append(fiab)
    fiabilidad_semanal = pd.concat(fiabilidad)
    guardarExcel(fiabilidad_semanal, "data/FiabilidadEstaciones.xlsx", append_sheet=False)

else:
    print("2")
    fiabilidad_semanal = pd.read_excel("data/FiabilidadEstaciones.xlsx")
    fiabilidad_semanal["Código"] = fiabilidad["Código"].astype(str).apply(rellenarId)


### Resumen Fiabilidad Vías

In [ ]:
fuentes_vias_df = (
    fuentes_vias[
        [
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Código",
            "Estación",
            "Vía Real de Estacionamiento",
            "Num. Trenes",
            "Registro Vía",
            "FuenteCTC",
            "FuenteSitra",
            "FuenteAger",
            "Coincide Vía",
            "Coincide Vía CTC",
            # "%RegistroVíaCTC",
        ]
    ]
    .groupby(
        [
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Código",
            "Estación",
            "Vía Real de Estacionamiento",
        ]
    )
    .agg("sum")
    .reset_index()
    .copy()
)
fuentes_vias_df["% Coincide Vía"] = (
    fuentes_vias_df["Coincide Vía"] / fuentes_vias_df["Num. Trenes"]
)
fuentes_vias_df["% Coincide Vía CTC"] = fuentes_vias_df["Coincide Vía CTC"].astype(
    float
) / fuentes_vias_df["FuenteCTC"].astype(float)
fuentes_vias_df["% Registro Vía CTC"] = (
    fuentes_vias_df["FuenteCTC"] / fuentes_vias_df["Num. Trenes"]
)
fuentes_vias_df = fuentes_vias_df.dropna(
    subset=["Código", "Estación", "Vía Real de Estacionamiento"]
).fillna(0)
fuentes_vias_df["Vía Real de Estacionamiento"] = (
    fuentes_vias_df["Vía Real de Estacionamiento"].astype(int).astype(str)
)
# fuentes_vias_df.head()

In [ ]:
fuentes_vias_df_semanal = (
    fuentes_vias_semanal[
        [   
            "Fecha",
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Código",
            "Estación",
            "Vía Real de Estacionamiento",
            "Num. Trenes",
            "Registro Vía",
            "FuenteCTC",
            "FuenteSitra",
            "FuenteAger",
            "Coincide Vía",
            "Coincide Vía CTC",
            # "%RegistroVíaCTC",
        ]
    ]
    .groupby(
        [   
            "Fecha",
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Código",
            "Estación",
            "Vía Real de Estacionamiento",
        ]
    )
    .agg("sum")
    .reset_index()
    .copy()
)
fuentes_vias_df_semanal["% Coincide Vía"] = (
    fuentes_vias_df_semanal["Coincide Vía"] / fuentes_vias_df_semanal["Num. Trenes"]
)
fuentes_vias_df_semanal["% Coincide Vía CTC"] = fuentes_vias_df_semanal["Coincide Vía CTC"].astype(
    float
) / fuentes_vias_df_semanal["FuenteCTC"].astype(float)
fuentes_vias_df_semanal["% Registro Vía CTC"] = (
    fuentes_vias_df_semanal["FuenteCTC"] / fuentes_vias_df_semanal["Num. Trenes"]
)
fuentes_vias_df_semanal = fuentes_vias_df_semanal.dropna(
    subset=["Código", "Estación", "Vía Real de Estacionamiento"]
).fillna(0)
fuentes_vias_df_semanal["Vía Real de Estacionamiento"] = (
    fuentes_vias_df_semanal["Vía Real de Estacionamiento"].astype(int).astype(str)
)

In [ ]:
ejemplo_fuentes_vias = (
    fuentes_vias_df
    # [
    #     (fuentes_vias_df["% Vía Reg."] == 1)
    #     & (fuentes_vias_df["% Coincide Vía Reg."] == 1)
    #     & (fuentes_vias_df["Num. Trenes"] > 100)
    # ]
    .sort_values(by=["Código", "Vía Real de Estacionamiento"]).reset_index(drop=True)
)
ejemplo_fuentes_vias["Código"] = ejemplo_fuentes_vias["Código"].apply(rellenarId)
ejemplo_fuentes_vias.head()

In [ ]:
ejemplo_fuentes_vias_semanal = (
    fuentes_vias_df_semanal
    # [
    #     (fuentes_vias_df["% Vía Reg."] == 1)
    #     & (fuentes_vias_df["% Coincide Vía Reg."] == 1)
    #     & (fuentes_vias_df["Num. Trenes"] > 100)
    # ]
    .sort_values(by=["Código", "Vía Real de Estacionamiento"]).reset_index(drop=True)
)
ejemplo_fuentes_vias_semanal["Código"] = ejemplo_fuentes_vias_semanal["Código"].apply(rellenarId)


In [ ]:
resumen_fiabilidad = pd.merge(
    ejemplo_fuentes_vias,
    fiabilidad,
    left_on=["Código", "Vía Real de Estacionamiento"],
    right_on=["Código", "Técnica"],
    how="left",
)

In [ ]:
resumen_fiabilidad_semanal = pd.merge(
    ejemplo_fuentes_vias_semanal,
    fiabilidad_semanal,
    left_on=["Código", "Vía Real de Estacionamiento"],
    right_on=["Código", "Técnica"],
    how="left")

In [ ]:
resumen_fiabilidad["% Registro Vía"] = (
    resumen_fiabilidad["Registro Vía"] / resumen_fiabilidad["Num. Trenes"]
)
resumen_fiabilidad = resumen_fiabilidad[
    [   
        "Provincia",
        "Subdirección",
        "Desc Delegacion/Gerencia PR",
        "Código",
        "Estación",
        "Vía Real de Estacionamiento",
        "Num. Trenes",
        "Registro Vía",
        "% Registro Vía",
        "Coincide Vía",
        "% Coincide Vía",
        "Técnica",
        "Comercial",
        "Fiabilidad",
    ]
].rename(columns={"Coincide Vía": "Correctas", "% Coincide Vía": "% Correctas"})

In [ ]:
resumen_fiabilidad_semanal["% Registro Vía"] = (
    resumen_fiabilidad_semanal["Registro Vía"] / resumen_fiabilidad_semanal["Num. Trenes"]
)
resumen_fiabilidad_semanal = resumen_fiabilidad_semanal[
    [
        "Fecha",
        "Provincia",
        "Subdirección",
        "Desc Delegacion/Gerencia PR",
        "Código",
        "Estación",
        "Vía Real de Estacionamiento",
        "Num. Trenes",
        "Registro Vía",
        "% Registro Vía",
        "Coincide Vía",
        "% Coincide Vía",
        "Técnica",
        "Comercial",
        "Fiabilidad",
    ]
].rename(columns={"Coincide Vía": "Correctas", "% Coincide Vía": "% Correctas"})

In [ ]:
# >90% fiabilidad no establecido como fiable
conf = 0.9
resumen_fiabilidad[
    (
        (resumen_fiabilidad["% Correctas"] >= conf)
        # (resumen_fiabilidad["% Coincide Vía CTC"] >= conf)
        # & (resumen_fiabilidad["% Registro Vía CTC"] >= conf)
    )
    # & (resumen_fiabilidad["Fiabilidad"].isna())
].sort_values(
    by=["Num. Trenes", "Código", "Vía Real de Estacionamiento"], ascending=False
).dropna().head()

In [ ]:
# <50% fiabilidad establecido como fiable
conf = 0.5
resumen_fiabilidad[
    (
        (resumen_fiabilidad["% Correctas"] <= conf)
        # | (resumen_fiabilidad["% Coincide Vía CTC"] <= conf)
        # | (resumen_fiabilidad["% Registro Vía CTC"] <= conf)
    )
    & np.invert(resumen_fiabilidad["Fiabilidad"].isna())
].sort_values(
    by=["Num. Trenes", "Código", "Vía Real de Estacionamiento"], ascending=False
).dropna().head()

### Detalle Vías

$$
precision_{vía} = \frac{tp}{tp+fp} = \frac{tp}{vía_{teorica}=v}
$$
$$
recall_{vía} = \frac{tp}{tp+fn} = \frac{tp}{vía_{real}=v}
$$
$$
F1_{vía} = harmonic\_mean(precision,recall) = 2\frac{precision_{vía}*recall_{vía}}{precision_{vía}+recall_{vía}}
$$


In [ ]:
# Estimación
def F1Score(tp: int, v_teorica: int, v_real: int):
    with np.errstate(divide="ignore", invalid="ignore"):
        precision = np.nan_to_num(tp / v_teorica, nan=0.0, posinf=0.0, neginf=0.0)
        recall = np.nan_to_num(tp / v_real, nan=0.0, posinf=0.0, neginf=0.0)

        f1 = statistics.harmonic_mean([precision, recall])
        # f1 = np.nan_to_num(
        #     2 * (precision * recall) / (precision + recall),
        #     nan=0.0,
        #     posinf=0.0,
        #     neginf=0.0,
        # )
    return precision, recall, f1

In [ ]:
estado_vias = []
resumen_estacion = []
for prov, subd, ger, cod, est in tqdm(
    detalle_tren[
        [
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Cod. Est",
            "Estación",
        ]
    ]
    .dropna()
    .drop_duplicates()
    .values
):
    # print(prov, subd, ger, cod, est)
    aux_est = (
        detalle_tren[
            (detalle_tren["Provincia"] == prov)
            & (detalle_tren["Subdirección"] == subd)
            & (detalle_tren["Desc Delegacion/Gerencia PR"] == ger)
            & (detalle_tren["Cod. Est"] == cod)
            & (detalle_tren["Estación"] == est)
        ]
        # .dropna(subset=["Vía Teórica", "Vía Real"])
        .reset_index(drop=True).copy()
    )
    aux_est[["Vía Teórica", "Vía Real"]] = (
        aux_est[["Vía Teórica", "Vía Real"]]
        .astype(str)
        .map(lambda x: x.split(".")[0].replace("nan", ""))
    )
    # aux_est[["Vía Teórica", "Vía Real"]] = (
    #     aux_est[["Vía Teórica", "Vía Real"]]
    #     .fillna("")
    #     .astype(str)
    #     .map(lambda x: x.split(".")[0].replace("nan", ""))
    # )
    # Estado Vías
    est_vias = []
    vias = set(aux_est["Vía Real"].tolist() + aux_est["Vía Teórica"].tolist())
    for v in vias:
        if isEmpty(v):
            continue
        tp = (aux_est["CoincideVía"]) & (aux_est["Vía Teórica"] == v)
        v_teorica = aux_est["Vía Teórica"] == v
        v_real = aux_est["Vía Real"] == v
        est_vias.append(
            [prov, subd, ger, cod, est, v, tp.sum(), v_teorica.sum(), v_real.sum()]
        )
    est_vias = pd.DataFrame(
        est_vias,
        columns=[
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Código",
            "Estación",
            "Vía",
            "tp",
            "v_teorica",
            "v_real",
        ],
    )
    est_vias[["Precisión", "Exhaustividad", "F1"]] = (
        est_vias[["tp", "v_teorica", "v_real"]]
        .apply(lambda x: F1Score(**x), axis=1)
        .tolist()
    )
    est_vias = est_vias.sort_values(by="Vía", key=np.int64)
    estado_vias.append(est_vias)

    # Resumen estación
    # Exactitud
    acc = aux_est["CoincideVía"].sum() / aux_est.shape[0]
    # # Micro_f1
    # F1Score(est_vias["tp"].sum(),est_vias["v_teorica"].sum(),est_vias["v_real"].sum())
    # Weighted f1
    with np.errstate(divide="ignore", invalid="ignore"):
        f1_weight = np.nan_to_num(
            (est_vias["F1"] * est_vias["v_real"]).sum() / est_vias["v_real"].sum(),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
    resumen_estacion.append(
        [prov, subd, ger, cod, est, est_vias["v_real"].sum(), acc, f1_weight]
    )

estado_vias = pd.concat(estado_vias)
resumen_estacion = pd.DataFrame(
    resumen_estacion,
    columns=[
        "Provincia",
        "Subdirección",
        "Desc Delegacion/Gerencia PR",
        "Código",
        "Estación",
        "MovimientosReales",
        "Exactitud",
        "F1_proporcional",
    ],
)

In [ ]:
estado_vias_semanal = []
resumen_estacion_semanal = []
for fec, prov, subd, ger, cod, est in tqdm(
    detalle_tren_semanal[
        [   
            "Fecha Origen",
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Cod. Est",
            "Estación",
        ]
    ]
    .dropna()
    .drop_duplicates()
    .values
):
    # print(prov, subd, ger, cod, est)
    aux_est_semanal = (
        detalle_tren_semanal[
            (detalle_tren_semanal["Fecha Origen"] == fec)&
            (detalle_tren_semanal["Provincia"] == prov)
            & (detalle_tren_semanal["Subdirección"] == subd)
            & (detalle_tren_semanal["Desc Delegacion/Gerencia PR"] == ger)
            & (detalle_tren_semanal["Cod. Est"] == cod)
            & (detalle_tren_semanal["Estación"] == est)
        ]
        # .dropna(subset=["Vía Teórica", "Vía Real"])
        .reset_index(drop=True).copy()
    )
    aux_est_semanal[["Vía Teórica", "Vía Real"]] = (
        aux_est_semanal[["Vía Teórica", "Vía Real"]]
        .astype(str)
        .map(lambda x: x.split(".")[0].replace("nan", ""))
    )
    # aux_est_semanal[["Vía Teórica", "Vía Real"]] = (
    #     aux_est_semanal[["Vía Teórica", "Vía Real"]]
    #     .fillna("")
    #     .astype(str)
    #     .map(lambda x: x.split(".")[0].replace("nan", ""))
    # )
    # Estado Vías
    est_vias_semanal = []
    vias_semanal = set(aux_est_semanal["Vía Real"].tolist() + aux_est_semanal["Vía Teórica"].tolist())
    for v in vias_semanal:
        if isEmpty(v):
            continue
        tp_semanal = (aux_est_semanal["CoincideVía"]) & (aux_est_semanal["Vía Teórica"] == v)
        v_teorica_semanal = aux_est_semanal["Vía Teórica"] == v
        v_real_semanal = aux_est_semanal["Vía Real"] == v
        est_vias_semanal.append(
            [fec,prov, subd, ger, cod, est, v, tp_semanal.sum(), v_teorica_semanal.sum(), v_real_semanal.sum()]
        )
    est_vias_semanal = pd.DataFrame(
        est_vias_semanal,
        columns=[
            "Fecha Origen",
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Código",
            "Estación",
            "Vía",
            "tp",
            "v_teorica",
            "v_real",
        ],
    )
    est_vias_semanal[["Precisión", "Exhaustividad", "F1"]] = (
        est_vias_semanal[["tp", "v_teorica", "v_real"]]
        .apply(lambda x: F1Score(**x), axis=1)
        .tolist()
    )
    est_vias_semanal = est_vias_semanal.sort_values(by="Vía", key=np.int64)
    estado_vias_semanal.append(est_vias_semanal)

    # Resumen estación
    # Exactitud
    acc = aux_est_semanal["CoincideVía"].sum() / aux_est_semanal.shape[0]
    # # Micro_f1
    # F1Score(est_vias["tp"].sum(),est_vias["v_teorica"].sum(),est_vias["v_real"].sum())
    # Weighted f1
    with np.errstate(divide="ignore", invalid="ignore"):
        f1_weight = np.nan_to_num(
            (est_vias_semanal["F1"] * est_vias_semanal["v_real"]).sum() / est_vias_semanal["v_real"].sum(),
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )
    resumen_estacion_semanal.append(
        [fec,prov, subd, ger, cod, est, est_vias_semanal["v_real"].sum(), acc, f1_weight]
    )

estado_vias_semanal = pd.concat(estado_vias_semanal)
resumen_estacion_semanal = pd.DataFrame(
    resumen_estacion_semanal,
    columns=[
        "Fecha Origen",
        "Provincia",
        "Subdirección",
        "Desc Delegacion/Gerencia PR",
        "Código",
        "Estación",
        "MovimientosReales",
        "Exactitud",
        "F1_proporcional",
    ],
)

In [ ]:
# display(resumen_estacion[resumen_estacion["Código"] == "13120"])
# display(estado_vias[estado_vias["Código"] == "13120"])

In [ ]:
# display(resumen_estacion[resumen_estacion["Código"] == "98305"])
# display(estado_vias[estado_vias["Código"] == "98305"])

In [ ]:
# Evaluamos el estado de las estaciones asignando una nota
def scale(x):
    esc = -8
    prop = 20 * np.log10(x)
    return 1 + (-esc) / (esc - np.exp(-1 / esc * prop))


def estadoEstacion(exactitud: float, f1: float, ntrenes: int):
    if ntrenes == 0:
        return None
    if exactitud == 0 or f1 == 0:
        return 0

    return scale(ntrenes) * statistics.harmonic_mean([exactitud, f1])


# x = np.arange(1, 20000, 1)
# y = scale(x)
# go.Figure([go.Scatter(x=x, y=y)], layout=dict(xaxis_type="log"))

In [ ]:
resumen_estacion["Estado"] = resumen_estacion[
    ["Exactitud", "F1_proporcional", "MovimientosReales"]
].apply(
    lambda x: estadoEstacion(
        x["Exactitud"], x["F1_proporcional"], x["MovimientosReales"]
    ),
    axis=1,
)

In [ ]:
resumen_estacion_semanal["Estado"] = resumen_estacion_semanal[
    ["Exactitud", "F1_proporcional", "MovimientosReales"]
].apply(
    lambda x: estadoEstacion(
        x["Exactitud"], x["F1_proporcional"], x["MovimientosReales"]
    ),
    axis=1,
)

### Matriz Confusión

In [ ]:
df_confusion = (
    detalle_tren[
        [
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Cod. Est",
            "Estación",
            "Vía Teórica",
            "Vía Real",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)
df_confusion[["Vía Teórica", "Vía Real"]] = (
    df_confusion[["Vía Teórica", "Vía Real"]]
    .astype(str)
    .map(lambda x: x.split(".")[0].replace("nan", "SinVía"))
)
# df_confusion[["Vía Teórica", "Vía Real"]] = (
#     df_confusion[["Vía Teórica", "Vía Real"]]
#     .fillna("")
#     .astype(str)
#     .map(lambda x: x.split(".")[0].replace("nan", "SinVía"))
# )
df_confusion = df_confusion.groupby(
    by=df_confusion.columns.tolist(), as_index=False
).size()
df_confusion = df_confusion[
    np.invert((df_confusion[["Vía Real", "Vía Teórica"]] == "SinVía").any(axis=1))
]
df_confusion["CORRECTO"] = df_confusion["size"] * (
    df_confusion["Vía Real"] == df_confusion["Vía Teórica"]
).astype(int)
sort_vias = {
    v: i for i, v in enumerate(sortStrNumbers(df_confusion["Vía Real"].unique()))
}
df_confusion["_ord_vias"] = df_confusion["Vía Real"].apply(sort_vias.get)
df_confusion.rename(columns={"Cod. Est": "Código"}, inplace=True)

In [ ]:
df_confusion_semanal = (
    detalle_tren_semanal[
        [   "Fecha Origen",
            "Provincia",
            "Subdirección",
            "Desc Delegacion/Gerencia PR",
            "Cod. Est",
            "Estación",
            "Vía Teórica",
            "Vía Real",
        ]
    ]
    .reset_index(drop=True)
    .copy()
)
df_confusion_semanal[["Vía Teórica", "Vía Real"]] = (
    df_confusion_semanal[["Vía Teórica", "Vía Real"]]
    .astype(str)
    .map(lambda x: x.split(".")[0].replace("nan", "SinVía"))
)
# df_confusion_semanal[["Vía Teórica", "Vía Real"]] = (
#     df_confusion_semanal[["Vía Teórica", "Vía Real"]]
#     .fillna("")
#     .astype(str)
#     .map(lambda x: x.split(".")[0].replace("nan", "SinVía"))
# )
df_confusion_semanal = df_confusion_semanal.groupby(
    by=df_confusion_semanal.columns.tolist(), as_index=False
).size()
df_confusion_semanal = df_confusion_semanal[
    np.invert((df_confusion_semanal[["Vía Real", "Vía Teórica"]] == "SinVía").any(axis=1))
]
df_confusion_semanal["CORRECTO"] = df_confusion_semanal["size"] * (
    df_confusion_semanal["Vía Real"] == df_confusion_semanal["Vía Teórica"]
).astype(int)
sort_vias = {
    v: i for i, v in enumerate(sortStrNumbers(df_confusion_semanal["Vía Real"].unique()))
}
df_confusion_semanal["_ord_vias"] = df_confusion_semanal["Vía Real"].apply(sort_vias.get)
df_confusion_semanal.rename(columns={"Cod. Est": "Código"}, inplace=True)

In [ ]:
df_estaciones = df_confusion.groupby(
    ["Provincia", "Subdirección", "Código", "Estación"]
).agg({
    "size": "sum",
    "CORRECTO": "sum"
}).reset_index()
df_estaciones["Porcentaje_Total"] = df_estaciones["CORRECTO"] / df_estaciones["size"]

df_estaciones.rename(
    columns={
        "size": "Tamaño_Total", 
        "CORRECTO": "Correctos_Total",
        "Porcentaje:": "Porcentaje_Total"
    },
    inplace=True
)

df_estaciones.sort_values(by=["Porcentaje_Total"])

In [ ]:
df_estaciones_semanal = df_confusion_semanal.groupby(
    ["Fecha Origen","Provincia", "Subdirección", "Código", "Estación"]
).agg({
    "size": "sum",
    "CORRECTO": "sum"
}).reset_index()
df_estaciones_semanal["Porcentaje_Total"] = df_estaciones_semanal["CORRECTO"] / df_estaciones_semanal["size"]

df_estaciones_semanal.rename(
    columns={
        "size": "Tamaño_Total", 
        "CORRECTO": "Correctos_Total",
        "Porcentaje:": "Porcentaje_Total"
    },
    inplace=True
)

df_estaciones_semanal.sort_values(by=["Porcentaje_Total"])

In [ ]:
planificacion_region = (
    df_confusion.groupby(
        ["Subdirección", "Provincia", "Estación", "Vía Real", "Vía Teórica"]
    )
    .agg({"size": "sum", "CORRECTO": "sum"})
    .reset_index()
    .rename(columns={"size": "Total"})
)

planificacion_estaciones = (
    df_confusion[["Subdirección", "Código", "Estación", "size"]]
    .groupby(["Subdirección", "Código", "Estación"])
    .agg("sum")
    .reset_index()
    .sort_values(by=["size"], ascending=False)
    .reset_index(drop=True)
    .reset_index()
)


In [ ]:
planificacion_region_semanal = (
    df_confusion_semanal.groupby(
        ["Fecha Origen","Subdirección", "Provincia", "Estación", "Vía Real", "Vía Teórica"]
    )
    .agg({"size": "sum", "CORRECTO": "sum"})
    .reset_index()
    .rename(columns={"size": "Total"})
)

planificacion_estaciones_semanal = (
    df_confusion_semanal[["Subdirección", "Código", "Estación", "size"]]
    .groupby(["Subdirección", "Código", "Estación"])
    .agg("sum")
    .reset_index()
    .sort_values(by=["size"], ascending=False)
    .reset_index(drop=True)
    .reset_index()
)


In [ ]:
centro = planificacion_region_semanal[planificacion_region_semanal["Subdirección"] == "Sur"].copy()


In [ ]:
test = centro[centro["Estación"] == "JODAR-UBEDA"].copy()

In [ ]:
test.sort_values(by=["Fecha Origen"], inplace=True)

In [ ]:
test.reset_index(drop=True,inplace=True)

In [ ]:
a = test.copy()

In [ ]:

a['Coincide'] = a.apply(lambda row: row['Total'] if row['Vía Real'] == row['Vía Teórica'] else 0, axis=1)


In [ ]:
coincide_por_dia = a.groupby(["Fecha Origen","Vía Real"])['Coincide'].sum().reset_index()


In [ ]:
coincide_por_dia = a.groupby(["Fecha Origen","Vía Real"])['Coincide'].sum().reset_index()
a['NoCoincide'] = a.apply(lambda row: row['Total'] if row['Vía Real'] != row['Vía Teórica'] else 0, axis=1)
Nocoincide_por_dia = a.groupby(["Fecha Origen","Vía Real"])['NoCoincide'].sum().reset_index()


In [ ]:
final = pd.merge(
    coincide_por_dia,
    Nocoincide_por_dia,
    how="left",
    on=["Fecha Origen","Vía Real"]
)

In [ ]:
Nocoincide_por_dia = a.groupby(["Fecha Origen","Vía Real"])['NoCoincide'].sum().reset_index()

In [ ]:
final = pd.merge(
    coincide_por_dia,
    Nocoincide_por_dia,
    how="left",
    on=["Fecha Origen","Vía Real"]
)

In [ ]:
final.sort_values(by=["Fecha Origen","Vía Real"],inplace=True)

In [ ]:


# grouped = (
#     final
#     .groupby(['Fecha Origen', 'Vía Teórica'])[['Coincide', 'NoCoincide']]
#     .sum()
#     .reset_index()
# )

# grouped['Fecha Origen'] = pd.to_datetime(grouped['Fecha Origen'])

# # ---- Orden natural de Vía (1,2,10... en vez de 1,10,2) ----
# vias_unicas_str = grouped['Vía Teórica'].astype(str).unique().tolist()

# def nat_key(s: str):
#     # separa números/letras para ordenar "naturalmente"
#     return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]

# orden_vias = sorted(vias_unicas_str, key=nat_key)

# # ---- Completar grid Fecha x Vía para tener mismas vías en todas las fechas ----
# fechas_unicas = grouped['Fecha Origen'].drop_duplicates().sort_values()

# full_idx = pd.MultiIndex.from_product(
#     [fechas_unicas, orden_vias],
#     names=['Fecha Origen', 'Vía Teórica']
# )

# grouped = (
#     grouped
#     .assign(**{'Vía Teórica': grouped['Vía Teórica'].astype(str)})
#     .set_index(['Fecha Origen', 'Vía Teórica'])
#     .reindex(full_idx, fill_value=0)
#     .reset_index()
# )

# # ---- Ordenar por Fecha y por orden natural de Vía ----
# map_via_orden = {v: i for i, v in enumerate(orden_vias)}
# grouped['__via_order__'] = grouped['Vía Teórica'].map(map_via_orden).fillna(len(orden_vias))
# grouped = grouped.sort_values(['Fecha Origen', '__via_order__']).drop(columns='__via_order__')

# # ----------------------------------------------------------------------
# # CLAVE: Hacer único el nivel superior del eje multicategoría y alinear
# # exactamente el categoryarray con ese nivel.
# # ----------------------------------------------------------------------

# # Claves únicas y etiquetas de fecha
# grouped['fecha_key']   = grouped['Fecha Origen'].dt.strftime('%Y-%m-%d')  # único (nivel superior)
# grouped['fecha_label'] = grouped['Fecha Origen'].dt.strftime('%d-%m')      # amigable para hover

# # Vía como string estable
# grouped['via_str'] = grouped['Vía Teórica'].astype(str)

# # Orden explícito y Categorical ordenado
# orden_fechas = fechas_unicas.dt.strftime('%Y-%m-%d').tolist()  # MISMO formato que fecha_key

# grouped['fecha_key'] = pd.Categorical(grouped['fecha_key'], categories=orden_fechas, ordered=True)
# grouped['via_str']   = pd.Categorical(grouped['via_str'],   categories=orden_vias,  ordered=True)

# # Eje multicategoría: usar las claves ordenadas
# x_multi = [grouped['fecha_key'], grouped['via_str']]

# # customdata para hover (usar etiqueta amigable + vía)
# customdata = np.stack([grouped['fecha_label'].astype(str), grouped['via_str'].astype(str)], axis=-1)

# # --- Colores ---
# color_coincide = '#2E8B57'  # SeaGreen
# color_no_coincide = '#CD5C5C'  # IndianRed

# # --- Gráfico ---
# fig = go.Figure()

# # Coincide
# fig.add_bar(
#     name='Coincide',
#     x=x_multi,
#     y=grouped['Coincide'],
#     marker_color=color_coincide,
#     text=grouped['Coincide'].apply(lambda v: f'{v}' if v > 0 else ''),
#     textposition='inside',
#     textfont=dict(size=9, color='white'),
#     customdata=customdata,
#     hovertemplate='Fecha: %{customdata[0]}<br>%{customdata[1]}<br>Coincide: %{y}<extra></extra>'
# )

# # No Coincide
# fig.add_bar(
#     name='No Coincide',
#     x=x_multi,
#     y=grouped['NoCoincide'],
#     marker_color=color_no_coincide,
#     text=grouped['NoCoincide'].apply(lambda v: f'{v}' if v > 0 else ''),
#     textposition='inside',
#     textfont=dict(size=9, color='white'),
#     customdata=customdata,
#     hovertemplate='Fecha: %{customdata[0]}<br>%{customdata[1]}<br>No Coincide: %{y}<extra></extra>'
# )

# # --- Diseño ---
# fig.update_layout(
#     barmode='stack',  # apilado Coincide + No Coincide por (Fecha, Vía)
#     title={
#         'text': 'Seguimiento semanal coincidencia vía',
#         'x': 0.5,
#         'font': {'size': 16}
#     },
#     xaxis_title='Fecha  -  Vía Planificada',
#     yaxis_title='Nº de trenes',
#     xaxis=dict(
#         type='multicategory',
#         # Importante: el categoryarray debe coincidir EXACTAMENTE con el primer nivel (fecha_key)
#         categoryorder='array',
#         categoryarray=orden_fechas,
#         tickangle=0,
#         tickfont=dict(size=10),
#         showgrid=True,
#         gridwidth=1,
#         gridcolor='lightgray'
#     ),
#     yaxis=dict(
#         showgrid=True,
#         gridwidth=1,
#         gridcolor='lightgray',
#         zeroline=True,
#         zerolinewidth=1,
#         zerolinecolor='gray',
#         tickfont=dict(size=11)
#     ),
#     height=700,
#     width=1200,
#     margin=dict(b=140, l=80, r=180, t=80),
#     legend=dict(
#         orientation="v",
#         yanchor="top",
#         y=1,
#         xanchor="left",
#         x=1.02,
#         font=dict(size=10)
#     ),
#     plot_bgcolor='white',
#     paper_bgcolor='white',
#     bargap=0.25,        # separación entre grupos (fechas)
#     bargroupgap=0.08,   # separación entre vías dentro de cada fecha
#     uniformtext_minsize=8,
#     uniformtext_mode='hide',
#     font=dict(family="DejaVu Sans")  # fuente estable para exportación con kaleido
# )

# # Muestra interactiva
# fig.show()
# fig.write_image(
#     testPath / "test_semanal.jpeg",
#     format="jpeg",
#     width=1200,
#     height=700,
#     scale=2  # aumenta la nitidez y reduce diferencias con el render interactivo
# )


#### Diagrama por región

In [ ]:
# levels = ["Vía Real", "Estación", "Provincia", "Subdirección"]
# color_columns = ["CORRECTO", "Total"]
# value_column = "Total"
# top_node = "España"
# sd = "España"
# fig = mostrarConfusionTree(
#     planificacion_region,
#     f"Calidad planificación {sd}",
#     levels,
#     value_column,
#     color_columns,
#     top_node=top_node,
# )

# fig.update_layout(
#     margin=dict(t=50, l=25, r=25, b=25),
# )
# fig.show()

In [ ]:

def seguimiento_semanal(df: pd.DataFrame, est: str,):

    grouped = (
        final
        .groupby(['Fecha Origen', 'Vía Real'])[['Coincide', 'NoCoincide']]
        .sum()
        .reset_index()
    )

    grouped['Fecha Origen'] = pd.to_datetime(grouped['Fecha Origen'])

    # ---- Orden natural de Vía (1,2,10... en vez de 1,10,2) ----
    vias_unicas_str = grouped['Vía Real'].astype(str).unique().tolist()

    def nat_key(s: str):
        # separa números/letras para ordenar "naturalmente"
        return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)]

    orden_vias = sorted(vias_unicas_str, key=nat_key)

    # ---- Completar grid Fecha x Vía para tener mismas vías en todas las fechas ----
    fechas_unicas = grouped['Fecha Origen'].drop_duplicates().sort_values()

    full_idx = pd.MultiIndex.from_product(
        [fechas_unicas, orden_vias],
        names=['Fecha Origen', 'Vía Real']
    )

    grouped = (
        grouped
        .assign(**{'Vía Real': grouped['Vía Real'].astype(str)})
        .set_index(['Fecha Origen', 'Vía Real'])
        .reindex(full_idx, fill_value=0)
        .reset_index()
    )

    # ---- Ordenar por Fecha y por orden natural de Vía ----
    map_via_orden = {v: i for i, v in enumerate(orden_vias)}
    grouped['__via_order__'] = grouped['Vía Real'].map(map_via_orden).fillna(len(orden_vias))
    grouped = grouped.sort_values(['Fecha Origen', '__via_order__']).drop(columns='__via_order__')

    # ----------------------------------------------------------------------
    # CLAVE: Hacer único el nivel superior del eje multicategoría y alinear
    # exactamente el categoryarray con ese nivel.
    # ----------------------------------------------------------------------

    # Claves únicas y etiquetas de fecha
    grouped['fecha_key']   = grouped['Fecha Origen'].dt.strftime('%Y-%m-%d')  # único (nivel superior)
    grouped['fecha_label'] = grouped['Fecha Origen'].dt.strftime('%d-%m')      # amigable para hover

    # Vía como string estable
    grouped['via_str'] = grouped['Vía Real'].astype(str)

    # Orden explícito y Categorical ordenado
    orden_fechas = fechas_unicas.dt.strftime('%Y-%m-%d').tolist()  # MISMO formato que fecha_key

    grouped['fecha_key'] = pd.Categorical(grouped['fecha_key'], categories=orden_fechas, ordered=True)
    grouped['via_str']   = pd.Categorical(grouped['via_str'],   categories=orden_vias,  ordered=True)

    # Eje multicategoría: usar las claves ordenadas
    x_multi = [grouped['fecha_key'], grouped['via_str']]

    # customdata para hover (usar etiqueta amigable + vía)
    customdata = np.stack([grouped['fecha_label'].astype(str), grouped['via_str'].astype(str)], axis=-1)

    # --- Colores ---
    color_coincide = '#2E8B57'  # SeaGreen
    color_no_coincide = '#CD5C5C'  # IndianRed

    # --- Gráfico ---
    fig = go.Figure()

    # Coincide
    fig.add_bar(
        name='Coincide',
        x=x_multi,
        y=grouped['Coincide'],
        marker_color=color_coincide,
        text=grouped['Coincide'].apply(lambda v: f'{v}' if v > 0 else ''),
        textposition='inside',
        textfont=dict(size=9, color='white'),
        customdata=customdata,
        hovertemplate='Fecha: %{customdata[0]}<br>%{customdata[1]}<br>Coincide: %{y}<extra></extra>'
    )

    # No Coincide
    fig.add_bar(
        name='No Coincide',
        x=x_multi,
        y=grouped['NoCoincide'],
        marker_color=color_no_coincide,
        text=grouped['NoCoincide'].apply(lambda v: f'{v}' if v > 0 else ''),
        textposition='inside',
        textfont=dict(size=9, color='white'),
        customdata=customdata,
        hovertemplate='Fecha: %{customdata[0]}<br>%{customdata[1]}<br>No Coincide: %{y}<extra></extra>'
    )

    # --- Diseño ---
    fig.update_layout(
        barmode='stack',  # apilado Coincide + No Coincide por (Fecha, Vía)
        title={
            'text': 'Seguimiento semanal coincidencia vía',
            'x': 0.5,
            'font': {'size': 20}
        },
        xaxis_title='Fecha  -  Vía real',
        yaxis_title='Nº de trenes',
        xaxis=dict(
            type='multicategory',
            # Importante: el categoryarray debe coincidir EXACTAMENTE con el primer nivel (fecha_key)
            categoryorder='array',
            categoryarray=orden_fechas,
            tickangle=0,
            tickfont=dict(size=10),
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray'
        ),
        yaxis=dict(
            showgrid=True,
            gridwidth=1,
            gridcolor='lightgray',
            zeroline=True,
            zerolinewidth=1,
            zerolinecolor='gray',
            tickfont=dict(size=11)
        ),
        height=700,
        width=1200,
        margin=dict(b=140, l=80, r=180, t=80),
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02,
            font=dict(size=10)
        ),
        plot_bgcolor='white',
        paper_bgcolor='white',
        bargap=0.25,        # separación entre grupos (fechas)
        bargroupgap=0.08,   # separación entre vías dentro de cada fecha
        uniformtext_minsize=8,
        uniformtext_mode='hide',
        font=dict(family="DejaVu Sans")  # fuente estable para exportación con kaleido
    )

    # Muestra interactiva
    return fig

In [ ]:

def mostrarConfusionSankey(
    df: pd.DataFrame, title: str, origen: str, destino: str, size: str
):
    """
    Genera un diagrama de sankey para confusión

    origen: str
        Columna origen
    destino: str
        Columna destino
    size: str
        Columna con el tamaño de la relación
    """
    df = df.rename(columns={size: "Total"}).copy()

    # Origen
    group_src = (
        df[[origen, destino, "Total"]]
        .groupby(origen)
        .agg({destino: "size", "Total": "sum"})
        .reset_index()
        .rename(columns={destino: "Destinos"})
        .copy()
    )
    _ord_map = {k: v for v, k in enumerate(sortElements(group_src[origen].values))}
    group_src["_ord"] = group_src[origen].apply(_ord_map.get)
    group_src = group_src.sort_values(by="_ord").reset_index(drop=True)
    group_src["_c_sum"] = group_src["Total"].cumsum()
    group_src["x_pos"] = 0.2
    group_src["y_pos"] = range_normalization(group_src["_c_sum"])
    hover_cols = [origen, "Destinos", "Total"]
    group_src["hover_text"] = setHoverInfo(group_src, hover_cols)
    group_src["label"] = group_src.apply(lambda x: f"<br>Vía{x[origen]}: {x['Total']:,}", axis=1)

    # Destino
    group_dst = (
        df[[origen, destino, "Total"]]
        .groupby(destino)
        .agg({origen: "size", "Total": "sum"})
        .reset_index()
        .rename(columns={origen: "Orígenes"})
        .copy()
    )
    _ord_map = {k: v for v, k in enumerate(sortElements(group_dst[destino].values))}
    group_dst["_ord"] = group_dst[destino].apply(_ord_map.get)
    group_dst = group_dst.sort_values(by="_ord").reset_index(drop=True)
    group_dst["_c_sum"] = group_dst["Total"].cumsum()
    group_dst["x_pos"] = 0.8
    group_dst["y_pos"] = range_normalization(group_dst["_c_sum"])
    hover_cols = [destino, "Orígenes", "Total"]
    group_dst["hover_text"] = setHoverInfo(group_dst, hover_cols)
    group_dst["label"] = group_dst.apply(lambda x: f"<br>Vía{x[destino]}: {x['Total']:,}", axis=1)

    # Nodos
    nodes = pd.concat(
        [
            group_src[[origen, "x_pos", "y_pos", "hover_text","label"]].rename(
                columns={origen: "Elemento"}
            ),
            group_dst[[destino, "x_pos", "y_pos", "hover_text","label"]].rename(
                columns={destino: "Elemento"}
            ),
        ],
        ignore_index=True,
    ).reset_index()
    elementos = nodes["Elemento"].unique()
    cmap_elementos = sample_random_colors(elementos)
    nodes["color"] = nodes["Elemento"].apply(cmap_elementos.get)

    # Enlaces
    hover_cols = [origen, destino, "Total"]
    hover_text_link = setHoverInfo(df, hover_cols)

    map_group_src = {
        k: v for v, k in enumerate(sortElements(group_src[origen].unique().tolist()))
    }
    map_group_dst = {
        k: v
        for v, k in enumerate(
            sortElements(group_dst[destino].unique().tolist()),
            start=len(map_group_src),
        )
    }

    traces = [
        go.Sankey(
            arrangement="snap",
            node=dict(
                pad=10,
                thickness=30,
                label=nodes["label"],
                x=nodes["x_pos"],
                y=nodes["y_pos"],
                color=nodes["color"],
                align="left",    
                customdata=nodes["hover_text"],
                hovertemplate="%{customdata}",
            ),
            link=dict(
                arrowlen=20,
                source=df[origen].apply(map_group_src.get),
                target=df[destino].apply(map_group_dst.get),
                value=df["Total"],  # .apply(np.log10),
                customdata=hover_text_link,
                hovertemplate="%{customdata}",
                hovercolor=[cmap_elementos[v] for v in df[origen]],
            ),
        )
    ]

    layout = setLayout(
        "togglegroup",
        title=title,
    )
    annotations = [
        {
            "xref": "paper",
            "yref": "paper",
            "x": 0.17,
            "y": -0.2,
            "text": "Planificación",
            "showarrow": False,
            "font": {"size": 15, "color": "black"},
        },
        {
            "xref": "paper",
            "yref": "paper",
            "x": 0.81,
            "y": -0.2,
            "text": "Real",
            "showarrow": False,
            "font": {"size": 15, "color": "black"},
        },
    ]
    fig = go.Figure(data=traces, layout=layout)
    fig = fig.update_layout(
        autosize=False,
        width=1200,
        height=500,
        title_x=0.5,
        margin=dict(l=0, r=0),
        annotations=annotations,
    )
    return fig


#### Diagrama por estación

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path

import plotly.io as pio
info_path = Path(r"C:\Users\xiangzhou.zhang\Documents\TEST\CoincidenciaVia")
fecha_ini =  (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
levels = ["Vía Teórica", "Vía Real", "Estación", "Provincia"]
color_columns = ["CORRECTO", "Total"]
value_column = "Total"
provincias= {"Centro":set(),"Sur":set(),"Norte":set(), "Noroeste":set(), "Noreste":set(), "Este":set(),"Alta Velocidad":set()}
try:
    pio.kaleido.scope._shutdown_kaleido()
except Exception:
    pass

pio.kaleido.scope.default_width = 1200
pio.kaleido.scope.default_height = 700
pio.kaleido.scope.default_scale = 2
for sd in planificacion_estaciones["Subdirección"].unique():
    # Ubicación archivos
    riv_path = info_path / str(sd)
    riv_path.mkdir(exist_ok=True, parents=True)
    import shutil
    try:
        if riv_path.exists():
            for p in riv_path.iterdir():
                try:
                    if p.is_file() or p.is_symlink():
                        p.unlink()
                    elif p.is_dir():
                        shutil.rmtree(p)
                except Exception as e:
                    print(f"Warning cleaning {p}: {e}")
    except Exception as e:
        print(f"Warning cleaning directory {riv_path}: {e}")

    # Top estaciones
    df_sd: pd.DataFrame = df_estaciones[
        df_estaciones["Subdirección"] == sd
    ].sort_values(by=["Porcentaje_Total"], ascending=True)
    fig_plan_reg = mostrarConfusionTree(
        planificacion_region[planificacion_region["Subdirección"] == sd],
        f"Calidad planificación {sd} {fecha_ini} - {fecha_fin}",
        levels,
        value_column,
        color_columns,
        top_node=sd,
    )
    # # fig_plan_reg.write_image(
    # #     riv_path / f"calidad_planificación_hoy.png", format="png", engine="kaleido"
    # # )
    # # fig_plan_reg.write_html(riv_path / f"calidad_planificación_hoy.html")
    # # # fig_plan_reg.show()

    # Planificación específica por estación
    for cod, est in df_sd[["Código", "Estación"]][:5].values:
        aux_df = df_confusion[
            (df_confusion["Código"] == cod)
            & (df_confusion["Estación"] == est)
            & (df_confusion["Subdirección"] == sd)
        ].copy()
        provincias[sd].add(est)
        fig = mostrarConfusionSankey(
            aux_df,
            f"Planificación Vías {est} ({"SD "+ sd}) {fecha_ini}",
            "Vía Teórica",
            "Vía Real",
            "size",
        )
        # fig.show
        fig.write_html(riv_path / f"{est}_hoy.html")
        # fig.show()
        semanal = planificacion_region_semanal[
            (planificacion_region_semanal["Estación"] == est)
            & (planificacion_region_semanal["Subdirección"] == sd)].copy()
        semanal['Coincide'] = semanal.apply(lambda row: row['Total'] if row['Vía Real'] == row['Vía Teórica'] else 0, axis=1)
        coincide_por_dia = semanal.groupby(["Fecha Origen","Vía Real"])['Coincide'].sum().reset_index()
        semanal['NoCoincide'] = semanal.apply(lambda row: row['Total'] if row['Vía Real'] != row['Vía Teórica'] else 0, axis=1)
        Nocoincide_por_dia = semanal.groupby(["Fecha Origen","Vía Real"])['NoCoincide'].sum().reset_index()
        final = pd.merge(
            coincide_por_dia,
            Nocoincide_por_dia,
            how="left",
            on=["Fecha Origen","Vía Real"]
        )
        fig1 = seguimiento_semanal(final,est)
        # fig1.write_image(riv_path / f"{est}_semanal.png", format="png")
        fig1.write_html(
            riv_path / f"{est}_semanal.html")

        # fig.write_html(riv_path / f"{est}_hoy.html")


    # guardarExcel(
    #     pd.merge(
    #         df_confusion[df_confusion["Subdirección"] == sd],
    #         df_sd.drop(["size"], axis=1),
    #         on=["Subdirección", "Cod. Est", "Estación"],
    #     )
    #     .sort_values(by=["index", "_ord_vias", "size"], ascending=[True, True, False])
    #     .drop(["_ord_vias", "index"], axis=1),
    #     riv_path / f"{fecha_ini} - {fecha_fin}.xlsx",
    #     sheet_name="Planificación",
    #     append_sheet=False,
    # )
    # guardarExcel(
    #     pd.merge(
    #         resumen_fiabilidad[resumen_fiabilidad["Subdirección"] == sd],
    #         df_sd.drop(["size"], axis=1),
    #         left_on=["Subdirección", "Código", "Estación"],
    #         right_on=["Subdirección", "Cod. Est", "Estación"],
    #     )
    #     .sort_values(by=["index", "Num. Trenes"], ascending=[True, False])
    #     .drop(["Cod. Est", "index"], axis=1),
    #     riv_path / f"{fecha_ini} - {fecha_fin}.xlsx",
    #     sheet_name="Fiabilidad",
    #     append_sheet=True,
    #)

In [ ]:


def html_to_png_precise(driver, html_file_path, output_png_path):
    driver.set_window_size(1200, 700)  # 👈 tamaño fijo

    driver.get(f"file:///{html_file_path}")
    time.sleep(2)  # esperar renderizado

    screenshot = driver.get_screenshot_as_png()
    with open(output_png_path, "wb") as f:
        f.write(screenshot)

    print(f"✅ {output_png_path}")

In [ ]:
def convert_all_html_to_png(base_path):
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")

    driver = webdriver.Chrome(options=chrome_options)

    html_files = list(base_path.rglob("*.html"))
    print(f"🔍 Encontrados {len(html_files)} archivos HTML")

    for html_file in html_files:
        html_path = html_file.resolve()
        png_path = html_path.with_suffix(".png")

        try:
            html_to_png_precise(driver, html_path, png_path)
        except Exception as e:
            print(f"❌ Error en {html_path}: {e}")

    driver.quit()
    print("🎉 Proceso terminado")

In [ ]:
info_path = Path(r"C:\Users\xiangzhou.zhang\Documents\TEST\CoincidenciaVia")
convert_all_html_to_png(info_path)

In [ ]:
# aux = df_confusion[df_confusion["Código"] == "35607"].copy()
# est = aux["Estación"].iloc[0]
# fig = mostrarConfusionSankey(
#             aux,
#             f"Planificación Vías {est} ({"SD "+ "SD CENTRO"}) {fecha_ini}",
#             "Vía Teórica",
#             "Vía Real",
#             "size",
#         )
# semanal = planificacion_region_semanal[planificacion_region_semanal["Estación"] == "MOSTOLES - EL SOTO"].copy()
# semanal['Coincide'] = semanal.apply(lambda row: row['Total'] if row['Vía Real'] == row['Vía Teórica'] else 0, axis=1)
# coincide_por_dia = semanal.groupby(["Fecha Origen","Vía Real"])['Coincide'].sum().reset_index()
# semanal['NoCoincide'] = semanal.apply(lambda row: row['Total'] if row['Vía Real'] != row['Vía Teórica'] else 0, axis=1)
# Nocoincide_por_dia = semanal.groupby(["Fecha Origen","Vía Real"])['NoCoincide'].sum().reset_index()
# final = pd.merge(
#     coincide_por_dia,
#     Nocoincide_por_dia,
#     how="left",
#     on=["Fecha Origen","Vía Real"]
# )
# fig1 = seguimiento_semanal(final,est)
# test_path = Path(r"C:\Users\xiangzhou.zhang\Documents\TEST\Mostoles_El_soto_ayer.png")
# fig.write_image(
#             test_path,
#                 format="png",
#                 width=1200,
#                 height=700,
#                 scale=2)    

In [ ]:
from datetime import datetime, timedelta
from pathlib import Path
from reportlab.lib.units import inch

def add_header(canvas, doc):
    canvas.saveState()
    fecha_actual = (datetime.now() - timedelta(days=1)).strftime("%d-%m-%Y")

    # Logo
    try:
        logo_path = Path("data/logo.png")
        canvas.drawImage(str(logo_path), doc.leftMargin, doc.height + doc.topMargin - 0.75*inch,
                         width=2*inch, height=0.75*inch, preserveAspectRatio=True)
    except:
        canvas.setFont("Helvetica", 10)
        canvas.drawString(doc.leftMargin, doc.height + doc.topMargin - 0.5*inch, "[LOGO NO ENCONTRADO]")

    # Posición base del título
    center_x = doc.width / 2.0 + doc.leftMargin
    y = doc.height + doc.topMargin - 0.4*inch

    # Títulos centrados
    canvas.setFont("Helvetica-Bold", 10)
    canvas.drawCentredString(center_x, y, "INDICADORES CALIDAD MSE")
    canvas.drawCentredString(center_x, y - 12, "Fiabilidad de vías")
    canvas.drawCentredString(center_x, y - 24, f"Análisis de datos {fecha_actual}")

    # Fecha a la derecha
    canvas.setFont("Helvetica", 10)
    canvas.drawRightString(doc.width + doc.leftMargin, doc.height + doc.topMargin - 0.3*inch,
                           f"Fecha: {fecha_actual}")

    canvas.restoreState()


In [ ]:
def add_footer(canvas, doc):
        canvas.saveState()
        
        # Obtener fecha actual
        fecha = (datetime.now() - timedelta(days=1)).strftime("%d-%m-%Y")
        styles = getSampleStyleSheet()
        
        # Configurar pie de página
        footer_text_left = "SD. de Sistemas y Medios Operacionales<br/>D. de Circulación y Gestión de Capacidad<br/>DG. de OPERACIONES Y EXPLOTACIÓN"
        footer_text_center = f"Página {doc.page}"
        footer_text_right = f"Fecha: {fecha}"
        footer_style = ParagraphStyle(
        'Footer',
        parent=styles['Normal'],
        fontSize=7,
        leading=10,
        spaceBefore=5,
        alignment=0  # Alineación izquierda
    )
        left_paragraph = Paragraph(footer_text_left, footer_style)
        left_paragraph.wrapOn(canvas, 3.5*inch, 0.5*inch)
        left_paragraph.drawOn(canvas, 0.5*inch, 0.3*inch)
        canvas.setFont("Helvetica", 9)
    
        
        # Centro (calculamos la posición central)
        text_width = canvas.stringWidth(footer_text_center, "Helvetica", 9)
        canvas.drawString((doc.width + doc.leftMargin + doc.rightMargin - text_width) / 2, 
                          0.5*inch, footer_text_center)
        
        # Derecha
        right_pos = doc.width + doc.leftMargin - canvas.stringWidth(footer_text_right, "Helvetica", 9) - 0.75*inch
        canvas.drawString(right_pos, 0.5*inch, footer_text_right)
        
        # Línea separadora
        canvas.setStrokeColor(colors.gray)
        canvas.setLineWidth(0.5)
        # canvas.line(0.75*inch, 0.7*inch, doc.width + doc.leftMargin - 0.75*inch, 0.7*inch)
        
        canvas.restoreState()

In [ ]:
def add_header_footer(canvas, doc):
    add_header(canvas, doc)
    add_footer(canvas, doc)

In [ ]:
import os
bookmarks= {}
def fiabilidad_PDF():
    """Versión con encabezado usando tabla e incluyendo imagen logo"""
    fecha_ayer = datetime.now() - timedelta(days=1)
    fecha_formateada = fecha_ayer.strftime("%Y-%m-%d")
    carpeta_destino = r"C:\Users\xiangzhou.zhang\ADIF\Elcano - Documentos\00-CALIDAD DATO\_Análisis Calidad Datos MSE y MIE\00.Rotulación-Fiabilidad-Supresiones\Fiabilidad"
    os.makedirs(carpeta_destino, exist_ok=True)
    semana = fecha_ayer.isocalendar()[1]
    filename = os.path.join(carpeta_destino, f"Semana{semana}_{fecha_formateada}_Fiabilidad_vía.pdf")

    doc = SimpleDocTemplate(filename, 
                            pagesize=A4,        
                            leftMargin=0.75*inch,
                            rightMargin=0.75*inch,
                            topMargin=1*inch,
                            bottomMargin=1*inch)
    
    styles = getSampleStyleSheet()
    story = []

    fecha_ayer = (datetime.now() - timedelta(days=1)).strftime("%d-%m-%Y")

    titulo_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontSize=15,
        spaceAfter=20,
        alignment=1,
        textColor=colors.black
    )
    story.append(Spacer(1, 70))
    # titulo = Paragraph("Análisis fiabilidad de vías " + fecha_ayer, titulo_style)
    # story.append(titulo)
    
    verde_oscuro = Color(0/255, 100/255, 0/255)
    verde_claro = Color(52/255, 207/255, 145/255) 
    barra_contenido = [
        ['ÍNDICE DE CONTENIDO']
    ]
    barra_descipción = [
        ['DESCRIPCIÓN']
    ]
    estilo_indice = TableStyle([
        ('BACKGROUND', (0,0), (-1,0), verde_oscuro),  
        ('TEXTCOLOR', (0,0), (-1,0), colors.white),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,0), 12),
        ('ALIGN', (0,0), (-1,-1), 'CENTER'),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('BOX', (0,0), (-1,-1), 1, colors.black),  
        ('INNERGRID', (0,0), (-1,-1), 0.5, colors.grey), 
        ('LEFTPADDING', (0,0), (-1,-1), 100),
        ('RIGHTPADDING', (0,0), (-1,-1), 100),
    ])
    estilo_descripción = TableStyle([
        ('BACKGROUND', (0,0), (-1,0), verde_claro),  
        ('TEXTCOLOR', (0,0), (-1,0), colors.white),
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,0), 9),
        ('ALIGN', (0,0), (-1,-1), 'CENTER'),
        ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
        ('BOX', (0,0), (-1,-1), 1, colors.black),  
        ('INNERGRID', (0,0), (-1,-1), 0.5, colors.grey),  
        ('LEFTPADDING', (0,0), (-1,-1), 100),
        ('RIGHTPADDING', (0,0), (-1,-1), 100),
    ])
    contenido_style = ParagraphStyle(
        'Contenido',
        parent=styles['Normal'],
        spaceAfter=12
    )
    contenido_center = ParagraphStyle(
        'ContenidoCenter',
        parent=styles['Normal'],
        # alignment=1,  # 1 = center
        spaceAfter=12
    )
 
    # for i in range(1, len(barra_contenido)):
    #     if i % 2 == 1:
    #         estilo_barra.add('BACKGROUND', (0,i), (-1,i), colors.lightgrey)
    #
    descripcion = Table(barra_descipción, colWidths=[6*inch])
    descripcion.setStyle(estilo_indice) 
    story.append(descripcion)   
    story.append(Spacer(1, 30))
    story.append(Paragraph("Este informe tiene por objeto de evaluar la fiabilidad en la asignación de vías de estacionamiento, mediante la comparación entre la vía planificada y la vía real de estacionamiento de cada circulación, según los datos registrados procedente de distintas fuentes(CTC, Sitra, Ager y Planificación).", contenido_style))
    story.append(Spacer(1, 30))
    indice_contenido = Table(barra_contenido, colWidths=[6*inch])
    indice_contenido.setStyle(estilo_indice)

    story.append(indice_contenido)
    story.append(Spacer(1,30))
    
    story.append(Paragraph("Análisis de coincidencia de vía SD CENTRO", contenido_center))
    story.append(Paragraph("Análisis de coincidencia de vía SD SUR", contenido_center))
    story.append(Paragraph("Análisis de coincidencia de vía SD NORTE", contenido_center))
    story.append(Paragraph("Análisis de coincidencia de vía SD NOROESTE", contenido_center))
    story.append(Paragraph("Análisis de coincidencia de vía SD NORESTE", contenido_center))
    story.append(Paragraph("Análisis de coincidencia de vía SD ESTE", contenido_center))
    story.append(Paragraph("Análisis de coincidencia de vía SD AV", contenido_center))
 

   
    
    # Indicadores
    story.append(PageBreak())
    story.append(Spacer(1, 300))
    # Construir el PDF
    Indicador_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontName='Helvetica',
        fontSize=20,
        spaceAfter=20,
        alignment=1,
        textColor=colors.black
    )
    Indicadores = Paragraph("Indicadores", Indicador_style)
    story.append(Indicadores)
    story.append(Spacer(1, 50))
    story.append(Paragraph("1. Planificación de vía: se muestra en gráficos, desglosados por subdirecciones, las cinco estaciones con mayor discrepancia entre la vía planificada y la vía real de llegada de los trenes.", contenido_style))
    story.append(Paragraph("2. Seguimiento Semanal: se muestra en gráficos, desglosados por subdirecciones, la coincidencia de las vías en los últimos 7 días  de las estaciones con mayor discrepancia", contenido_style))
    #graficos
    story.append(PageBreak())
  
    
    # Añadir gráfico de planificación de vía para la subdirección Centro
    
    ruta = Path(r"C:\Users\xiangzhou.zhang\Documents\TEST\CoincidenciaVia")

    # Para evitar ordenar aleatoriamente si 'imagenes_provincia' es un set, puedes ordenar:
    # imagenes = sorted(list(imagenes_provincia))

    provincias_keys = list(provincias.keys())

    for provincia_idx, (provincia_nombre, imagenes_provincia) in enumerate(provincias.items()):
        ruta_provincias = ruta / provincia_nombre
        print(f"Procesando imágenes para {ruta_provincias}...")

        imagenes = list(imagenes_provincia)
        if not imagenes:
            print(f"  - No hay imágenes para {provincia_nombre}")
            # Si no hay imágenes, pasamos a la siguiente provincia
            continue

        for i, base in enumerate(imagenes):
            # Rutas de la pareja de imágenes para la misma "vía"
            ruta_imagen_hoy = ruta_provincias / f"{base}_hoy.png"
            ruta_imagen_semanal = ruta_provincias / f"{base}_semanal.png"

            existe_hoy = ruta_imagen_hoy.exists()
            existe_semanal = ruta_imagen_semanal.exists()

            if not (existe_hoy or existe_semanal):
                print(f"  - No se encontraron imágenes para '{base}' en {provincia_nombre} (hoy/semanal)")
                # No añadimos página vacía; pasamos al siguiente
                continue

            # ---- NUEVA PÁGINA: encabezado + hoy + semanal ----
            story.append(Spacer(1, 0))  # pequeño margen superior

            barra_centro = [[f"Subdirección: {provincia_nombre}"]]
            centro = Table(barra_centro, colWidths=[6 * inch])
            centro.setStyle(estilo_indice)
            story.append(Spacer(1,70))
            story.append(centro)
            story.append(Spacer(1, 30))

            # Imagen HOY (si existe)
            if existe_hoy:
                story.append(Image(ruta_imagen_hoy, width=6 * inch, height=3.2 * inch))
                story.append(Spacer(1, 20))
            else:
                print(f"  - Imagen no encontrada: {ruta_imagen_hoy}")

            # Imagen SEMANAL (si existe)
            if existe_semanal:
                story.append(Image(ruta_imagen_semanal, width=6 * inch, height=3.2 * inch))
                story.append(Spacer(1, 20))
            else:
                print(f"  - Imagen no encontrada: {ruta_imagen_semanal}")

            # ---- Salto de página ----
            es_ultimo_item_en_provincia = (i == len(imagenes) - 1)
            es_ultima_provincia = (provincia_idx == len(provincias_keys) - 1)

            # Si NO es el último item de la provincia, siempre salto de página
            if not es_ultimo_item_en_provincia:
                story.append(PageBreak())
            # Si es el último item de la provincia pero NO es la última provincia, salto de página
            elif not es_ultima_provincia:
                story.append(PageBreak())

    # Construcción final del PDF
    doc.build(story, onFirstPage=add_header_footer, onLaterPages=add_header_footer)
    print(f"PDF con encabezado en tabla creado: {filename}")


In [ ]:
fiabilidad_PDF()
